In [1]:
"""
## Notebook 2 of 2 — Fairness Audit (Google Colab)

This notebook contains the Lecture 3 fairness audit using solas-ai v0.6.0.
It is a companion to RML_Capstone_Main.ipynb.

solas-ai is not available on PyPI for Windows local environments and 
requires Google Colab to run.

To reproduce:
1. Run RML_Capstone_Main.ipynb locally to generate hmda_cleaned.parquet
2. Upload hmda_cleaned.parquet to Google Drive
3. Run this notebook on Google Colab

The hmda_cleaned.parquet file is generated by the cleaning pipeline 
in RML_Capstone_Main.ipynb from the raw 2024 HMDA LAR file.
Professor: replace CLEANED_DATA_PATH below with your Drive path.
"""

CLEANED_DATA_PATH = '/content/drive/MyDrive/hmda_cleaned.parquet'

# Fairness Audit (using solas-ai)

In [3]:
!pip install solas-ai

ERROR: Could not find a version that satisfies the requirement solas-ai (from versions: none)
ERROR: No matching distribution found for solas-ai


In [2]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_parquet(CLEANED_DATA_PATH)
print(df.shape)

SyntaxError: invalid syntax (599815661.py, line 2)

In [6]:
# ── Step 1: Mount Drive and load data ────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import xgboost as xgb

df = pd.read_parquet('/content/drive/MyDrive/hmda_cleaned.parquet')
print(df.shape)

# ── Step 2: Feature lists ─────────────────────────────────────
numeric_features = [
    'income', 'property_value',
    'tract_to_msa_income_percentage',
    'ffiec_msa_md_median_family_income'
]

categorical_features = [
    'dti_bucket', 'loan_purpose_factor', 'loan_type_factor',
    'occupancy_factor', 'lien_factor', 'submission_factor',
    'dwelling_factor', 'state_factor'
]

# ── Step 3: Train/test split ──────────────────────────────────
X = df[numeric_features + categorical_features]
y = df['target'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# ── Step 4: Preprocessor ─────────────────────────────────────
preprocessor = ColumnTransformer([
    ("num", "passthrough", numeric_features),
    ("cat", OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
        encoded_missing_value=-2
    ), categorical_features)
])

# ── Step 5: Train final model ─────────────────────────────────
xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", xgb.XGBClassifier(
        objective='binary:logistic',
        n_estimators=200,
        max_depth=4,
        random_state=42,
        eval_metric='logloss',
        tree_method='hist',
        n_jobs=-1
    ))
])

print("Training...")
xgb_pipeline.fit(X_train, y_train)
print("Done.")

# ── Step 6: Predictions and threshold ────────────────────────
THRESHOLD = 0.70

y_pred_prob_final = xgb_pipeline.predict_proba(X_test)[:, 1]
y_pred_final      = (y_pred_prob_final >= THRESHOLD).astype(int)

# ── Step 7: X_test_copy and df_fairness ──────────────────────
X_test_copy = X_test.copy()
X_test_copy['actual']             = y_test.values
X_test_copy['pred']               = y_pred_final
X_test_copy['pred_prob']          = y_pred_prob_final
X_test_copy['derived_race']       = df.loc[X_test.index, 'derived_race'].values
X_test_copy['derived_sex']        = df.loc[X_test.index, 'derived_sex'].values
X_test_copy['derived_ethnicity']  = df.loc[X_test.index, 'derived_ethnicity'].values

df_fairness = X_test_copy[
    X_test_copy['derived_race'].notna() &
    X_test_copy['derived_sex'].notna() &
    X_test_copy['derived_ethnicity'].notna() &
    (X_test_copy['derived_race'] != 'Joint') &
    (X_test_copy['derived_sex'] != 'Joint')
].copy()

print(f"X_test_copy : {len(X_test_copy):,}")
print(f"df_fairness : {len(df_fairness):,}")
print(f"Pos rate    : {y_pred_final.mean():.4f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(8661795, 34)
Train: (6929436, 12), Test: (1732359, 12)
Training...
Done.
X_test_copy : 1,732,359
df_fairness : 837,552
Pos rate    : 0.7569


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8661795 entries, 0 to 8661794
Data columns (total 34 columns):
 #   Column                             Dtype   
---  ------                             -----   
 0   action_taken                       int64   
 1   derived_race                       object  
 2   derived_ethnicity                  object  
 3   derived_sex                        object  
 4   applicant_age                      object  
 5   derived_dwelling_category          object  
 6   tract_minority_population_percent  float64 
 7   tract_to_msa_income_percentage     float64 
 8   ffiec_msa_md_median_family_income  int64   
 9   derived_msa_md                     int64   
 10  occupancy_type                     int64   
 11  lien_status                        int64   
 12  submission_of_application          Int64   
 13  state_code                         object  
 14  income                             float64 
 15  debt_to_income_ratio               object  
 16  

In [10]:
THRESHOLD = 0.70

y_pred_prob_final = xgb_pipeline.predict_proba(X_test)[:, 1]
y_pred_final      = (y_pred_prob_final >= THRESHOLD).astype(int)

# Attach predictions AND all factor columns from df
X_test_copy = X_test.copy()
X_test_copy['actual']    = y_test.values
X_test_copy['pred']      = y_pred_final
X_test_copy['pred_prob'] = y_pred_prob_final

# Attach all protected attribute columns from df
for col in ['derived_race', 'derived_ethnicity', 'derived_sex',
            'applicant_age', 'race_factor', 'ethnicity_factor',
            'sex_factor', 'age_factor']:
    X_test_copy[col] = df.loc[X_test.index, col].values

# Fairness subset — exclude Joint and nulls on derived columns
df_fairness = X_test_copy[
    X_test_copy['derived_race'].notna() &
    X_test_copy['derived_sex'].notna() &
    X_test_copy['derived_ethnicity'].notna() &
    (X_test_copy['derived_race'] != 'Joint') &
    (X_test_copy['derived_sex'] != 'Joint')
].copy()

print(f"X_test_copy : {len(X_test_copy):,}")
print(f"df_fairness : {len(df_fairness):,}")
print(f"Columns: {df_fairness.columns.tolist()}")

X_test_copy : 1,732,359
df_fairness : 837,552
Columns: ['income', 'property_value', 'tract_to_msa_income_percentage', 'ffiec_msa_md_median_family_income', 'dti_bucket', 'loan_purpose_factor', 'loan_type_factor', 'occupancy_factor', 'lien_factor', 'submission_factor', 'dwelling_factor', 'state_factor', 'actual', 'pred', 'pred_prob', 'derived_race', 'derived_ethnicity', 'derived_sex', 'applicant_age', 'race_factor', 'ethnicity_factor', 'sex_factor', 'age_factor']


In [ ]:
import solas_disparity as sd
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

def air_me_by_group(data, outcome_col, group_col, reference_group, label, group_category):
    work = data[[group_col, outcome_col]].copy()
    work[outcome_col] = pd.to_numeric(work[outcome_col], errors="coerce")
    work = work.dropna(subset=[group_col, outcome_col]).copy()
    work[outcome_col] = work[outcome_col].astype(int)

    manual_table = (
        work.groupby(group_col, dropna=False)[outcome_col]
        .agg(["count", "sum", "mean"])
        .reset_index()
        .rename(columns={group_col:"group","count":"n",
                         "sum":"favorable","mean":"selection_rate"})
    )

    if reference_group not in manual_table["group"].values:
        raise ValueError(f"Reference group '{reference_group}' not found.")

    ref_rate = manual_table.loc[
        manual_table["group"] == reference_group, "selection_rate"
    ].iloc[0]

    manual_table["AIR"]     = (manual_table["selection_rate"] / ref_rate).round(4)
    manual_table["ME"]      = (manual_table["selection_rate"] - ref_rate).round(4)
    manual_table["flag_80"] = np.where(manual_table["AIR"] < 0.80, "*** BELOW 0.80", "")
    manual_table = manual_table.sort_values("AIR").reset_index(drop=True)

    group_dummies = pd.get_dummies(work[group_col], dtype=int)
    solas_results = []

    for grp in group_dummies.columns:
        if grp == reference_group:
            continue
        air_obj = sd.adverse_impact_ratio(
            group_data        = group_dummies[[grp, reference_group]],
            protected_groups  = [grp],
            reference_groups  = [reference_group],
            group_categories  = [group_category],
            outcome           = work[outcome_col],
            sample_weight     = None,
            air_threshold     = 0.80,
            percent_difference_threshold = 0.0,
        )
        tbl = air_obj.summary_table.copy().reset_index()
        if "Group" not in tbl.columns:
            tbl = tbl.rename(columns={tbl.columns[0]: "Group"})
        tbl = tbl[tbl["Group"] == grp].copy()
        tbl = tbl.rename(columns={
            "Group": "group", "Total": "n_solas",
            "Favorable": "favorable_solas",
            "Percent Favorable": "selection_rate_solas",
            "AIR": "AIR_solas",
            "Percent Difference Favorable": "ME_solas",
            "P-Values": "p_value",
            "Practically Significant": "practically_significant",
        })
        for col in ["selection_rate_solas", "ME_solas"]:
            if col in tbl.columns:
                tbl[col] = pd.to_numeric(tbl[col], errors="coerce")
                if tbl[col].abs().max() > 1:
                    tbl[col] = tbl[col] / 100.0
        solas_results.append(tbl)

    solas_table = pd.concat(solas_results, ignore_index=True) if solas_results else pd.DataFrame()
    combined = manual_table.merge(
        solas_table[['group','AIR_solas','p_value','practically_significant']],
        on='group', how='left'
    )

    print(f"\n{label}")
    print(combined.to_string(index=False))
    return combined

def error_rates_by_group(data, group_col, pred_col, outcome_col):
    results = []
    for grp, g in data.groupby(group_col, dropna=False):
        tp = ((g[pred_col]==1) & (g[outcome_col]==1)).sum()
        tn = ((g[pred_col]==0) & (g[outcome_col]==0)).sum()
        fp = ((g[pred_col]==1) & (g[outcome_col]==0)).sum()
        fn = ((g[pred_col]==0) & (g[outcome_col]==1)).sum()
        n  = len(g)
        fpr = fp/(fp+tn) if (fp+tn)>0 else np.nan
        fnr = fn/(fn+tp) if (fn+tp)>0 else np.nan
        results.append({
            group_col: grp, "n": n,
            "FPR": round(fpr,4), "FNR": round(fnr,4),
        })
    return pd.DataFrame(results).sort_values("n", ascending=False).reset_index(drop=True)

In [13]:
import warnings
warnings.filterwarnings('ignore')

# ── 1. Race ───────────────────────────────────────────────────
race_air = air_me_by_group(
    df_fairness, 'pred', 'derived_race', 'White',
    'AIR/ME by Race (reference=White)', 'race'
)
race_err = error_rates_by_group(df_fairness, 'derived_race', 'pred', 'actual')
print("\nFPR/FNR by Race:")
print(race_err.to_string(index=False))

# ── 2. Ethnicity ──────────────────────────────────────────────
eth_air = air_me_by_group(
    df_fairness, 'pred', 'derived_ethnicity', 'Not Hispanic or Latino',
    'AIR/ME by Ethnicity (reference=Not Hispanic or Latino)', 'ethnicity'
)
eth_err = error_rates_by_group(df_fairness, 'derived_ethnicity', 'pred', 'actual')
print("\nFPR/FNR by Ethnicity:")
print(eth_err.to_string(index=False))


AIR/ME by Race (reference=White)
                                    group      n  favorable  selection_rate    AIR      ME flag_80  AIR_solas       p_value practically_significant
Native Hawaiian or Other Pacific Islander   3006       1825        0.607119 0.8210 -0.1323           0.821025  7.192472e-61                      No
                Black or African American 114925      71461        0.621806 0.8409 -0.1177           0.840886  0.000000e+00                      No
         American Indian or Alaska Native   9895       6384        0.645174 0.8725 -0.0943           0.872488  2.418214e-99                      No
                 2 or more minority races   3153       2053        0.651126 0.8805 -0.0883           0.880536  2.396236e-29                      No
                                    White 638759     472340        0.739465 1.0000  0.0000                NaN           NaN                     NaN
                                    Asian  67814      54582        0.804878 1.

In [17]:
# ── Intersectional: Race x Ethnicity ─────────────────────────
df_fairness['race_ethnicity'] = (
    df_fairness['derived_race'] + " | " + df_fairness['derived_ethnicity']
)

intersect_race_eth = error_rates_by_group(
    df_fairness, 'race_ethnicity', 'pred', 'actual'
)

# Remove Joint ethnicity rows from intersectional table
intersect_race_eth_clean = intersect_race_eth[
    ~intersect_race_eth['race_ethnicity'].str.contains('Joint')
].copy()

print("\nFPR/FNR by Race x Ethnicity (Intersectional, Joint excluded):")
print(intersect_race_eth_clean.to_string(index=False))


FPR/FNR by Race x Ethnicity (Intersectional, Joint excluded):
                                                    race_ethnicity      n    FPR    FNR
                                    White | Not Hispanic or Latino 526257 0.3384 0.1262
                                        White | Hispanic or Latino 108974 0.3229 0.1294
                Black or African American | Not Hispanic or Latino 108575 0.2894 0.1839
                                    Asian | Not Hispanic or Latino  64926 0.3638 0.0647
         American Indian or Alaska Native | Not Hispanic or Latino   5924 0.2997 0.1512
                    Black or African American | Hispanic or Latino   5565 0.3194 0.1401
             American Indian or Alaska Native | Hispanic or Latino   3857 0.2821 0.1366
                                        Asian | Hispanic or Latino   2308 0.3170 0.0885
                 2 or more minority races | Not Hispanic or Latino   2267 0.3067 0.1258
Native Hawaiian or Other Pacific Islander | Not Hispanic 

In [18]:
# ── Sex ───────────────────────────────────────────────────────
sex_air = air_me_by_group(
    df_fairness, 'pred', 'sex_factor', 'Male',
    'AIR/ME by Sex (reference=Male)', 'sex'
)
sex_err = error_rates_by_group(df_fairness, 'sex_factor', 'pred', 'actual')
print("\nFPR/FNR by Sex:")
print(sex_err.to_string(index=False))

# ── Age ───────────────────────────────────────────────────────
age_air = air_me_by_group(
    df_fairness, 'pred', 'age_factor', '35-44',
    'AIR/ME by Age (reference=35-44)', 'age'
)
age_err = error_rates_by_group(df_fairness, 'age_factor', 'pred', 'actual')
print("\nFPR/FNR by Age:")
print(age_err.to_string(index=False))

# ── Intersectional: Race x Sex ────────────────────────────────
df_fairness['race_sex'] = (
    df_fairness['race_factor'].astype(str) + " | " +
    df_fairness['sex_factor'].astype(str)
)
intersect_race_sex = error_rates_by_group(
    df_fairness, 'race_sex', 'pred', 'actual'
)
intersect_race_sex_clean = intersect_race_sex[
    ~intersect_race_sex['race_sex'].str.contains('Joint')
].copy()

print("\nFPR/FNR by Race x Sex (Intersectional, Joint excluded):")
print(intersect_race_sex_clean.to_string(index=False))


AIR/ME by Sex (reference=Male)
 group      n  favorable  selection_rate    AIR      ME flag_80  AIR_solas  p_value practically_significant
Female 337955     235367        0.696445 0.9321 -0.0507           0.932125      0.0                      No
  Male 499597     373278        0.747158 1.0000  0.0000                NaN      NaN                     NaN
 Joint      0          0             NaN    NaN     NaN                NaN      1.0                        

FPR/FNR by Sex:
sex_factor      n    FPR    FNR
      Male 499597 0.3530 0.1179
    Female 337955 0.2929 0.1437
     Joint      0    NaN    NaN

AIR/ME by Age (reference=35-44)
group      n  favorable  selection_rate    AIR      ME        flag_80  AIR_solas       p_value practically_significant
  >74  34337      20471        0.596179 0.7889 -0.1595 *** BELOW 0.80   0.788893  0.000000e+00                     Yes
65-74  78939      49345        0.625103 0.8272 -0.1306                  0.827166  0.000000e+00                      No
5

In [19]:
# ── AIR/ME for Race x Sex (manual only, no solas-ai) ─────────
df_fairness['race_sex_clean'] = (
    df_fairness['race_factor'].astype(str) + " | " +
    df_fairness['sex_factor'].astype(str)
)

# Remove Joint
work = df_fairness[
    ~df_fairness['race_sex_clean'].str.contains('Joint')
].copy()

# Reference = White | Male
ref_rate = work[work['race_sex_clean'] == 'White | Male']['pred'].mean()

intersect_air = (
    work.groupby('race_sex_clean')['pred']
    .agg(['count','sum','mean'])
    .reset_index()
    .rename(columns={'race_sex_clean':'group','count':'n',
                     'sum':'favorable','mean':'selection_rate'})
)

intersect_air['AIR'] = (intersect_air['selection_rate'] / ref_rate).round(4)
intersect_air['ME']  = (intersect_air['selection_rate'] - ref_rate).round(4)
intersect_air['flag_80'] = np.where(intersect_air['AIR'] < 0.80, '*** BELOW 0.80', '')
intersect_air = intersect_air.sort_values('AIR').reset_index(drop=True)

print("\nAIR/ME by Race x Sex (reference=White | Male):")
print(intersect_air.to_string(index=False))


AIR/ME by Race x Sex (reference=White | Male):
                                             group      n  favorable  selection_rate    AIR      ME        flag_80
Native Hawaiian or Other Pacific Islander | Female   1079        608        0.563485 0.7455 -0.1923 *** BELOW 0.80
                Black or African American | Female  60853      36522        0.600168 0.7940 -0.1557 *** BELOW 0.80
                 2 or more minority races | Female   1406        850        0.604552 0.7999 -0.1513 *** BELOW 0.80
         American Indian or Alaska Native | Female   3920       2431        0.620153 0.8205 -0.1357               
  Native Hawaiian or Other Pacific Islander | Male   1927       1217        0.631552 0.8356 -0.1243               
                  Black or African American | Male  54072      34939        0.646157 0.8549 -0.1097               
           American Indian or Alaska Native | Male   5975       3953        0.661590 0.8753 -0.0942               
                   2 or more min

In [20]:
# ── Intersectional: Ethnicity x Sex ──────────────────────────
work_eth_sex = df_fairness[
    ~df_fairness['sex_factor'].astype(str).str.contains('Joint')
].copy()

work_eth_sex['eth_sex'] = (
    work_eth_sex['ethnicity_factor'].astype(str) + " | " +
    work_eth_sex['sex_factor'].astype(str)
)

# Reference = Not Hispanic or Latino | Male
ref_rate = work_eth_sex[
    work_eth_sex['eth_sex'] == 'Not Hispanic or Latino | Male'
]['pred'].mean()

intersect_eth_sex = (
    work_eth_sex.groupby('eth_sex')['pred']
    .agg(['count', 'sum', 'mean'])
    .reset_index()
    .rename(columns={'eth_sex': 'group', 'count': 'n',
                     'sum': 'favorable', 'mean': 'selection_rate'})
)

intersect_eth_sex['AIR']     = (intersect_eth_sex['selection_rate'] / ref_rate).round(4)
intersect_eth_sex['ME']      = (intersect_eth_sex['selection_rate'] - ref_rate).round(4)
intersect_eth_sex['flag_80'] = np.where(intersect_eth_sex['AIR'] < 0.80, '*** BELOW 0.80', '')
intersect_eth_sex = intersect_eth_sex.sort_values('AIR').reset_index(drop=True)

print("\nAIR/ME by Ethnicity x Sex (reference=Not Hispanic or Latino | Male):")
print(intersect_eth_sex.to_string(index=False))

# FPR/FNR
err_eth_sex = error_rates_by_group(work_eth_sex, 'eth_sex', 'pred', 'actual')
print("\nFPR/FNR by Ethnicity x Sex:")
print(err_eth_sex.to_string(index=False))


AIR/ME by Ethnicity x Sex (reference=Not Hispanic or Latino | Male):
                          group      n  favorable  selection_rate    AIR      ME flag_80
    Hispanic or Latino | Female  46012      31087        0.675628 0.8977 -0.0770        
                 Joint | Female   2171       1467        0.675725 0.8979 -0.0769        
Not Hispanic or Latino | Female 289772     202813        0.699905 0.9300 -0.0527        
      Hispanic or Latino | Male  76336      54763        0.717394 0.9532 -0.0352        
                   Joint | Male   3045       2263        0.743186 0.9875 -0.0094        
  Not Hispanic or Latino | Male 420216     316252        0.752594 1.0000  0.0000        

FPR/FNR by Ethnicity x Sex:
                        eth_sex      n    FPR    FNR
  Not Hispanic or Latino | Male 420216 0.3569 0.1172
Not Hispanic or Latino | Female 289772 0.2927 0.1442
      Hispanic or Latino | Male  76336 0.3355 0.1230
    Hispanic or Latino | Female  46012 0.2948 0.1415
             

fairness_audit_text = """
## Fairness Audit — Lecture 3

### Framework
The fairness audit follows the solas-ai disparity testing framework from Lecture 3.
Three primary metrics are computed:
- **AIR (Adverse Impact Ratio)**: ratio of selection rates between protected and
  reference group. EEOC 80% rule: AIR >= 0.80 required.
- **ME (Marginal Effect)**: absolute difference in selection rates.
- **FPR/FNR**: error rate disparities by group.

Audit conducted on df_fairness (n=837,552) — applicants with known single-identity
race, sex, and ethnicity. Joint applicants excluded from protected attribute analysis.

Reference groups: White (race), Not Hispanic or Latino (ethnicity),
Male (sex), 35-44 (age).

---

### 1. Race

| Group | n | Selection Rate | AIR | ME | Flag | FPR | FNR |
|---|---|---|---|---|---|---|---|
| White (ref) | 638,759 | 0.739 | 1.000 | 0.000 | | 0.335 | 0.127 |
| Asian | 67,814 | 0.805 | 1.089 | +0.065 | | 0.361 | 0.065 |
| 2+ minority | 3,153 | 0.651 | 0.881 | -0.088 | | 0.295 | 0.129 |
| AIAN | 9,895 | 0.645 | 0.873 | -0.094 | | 0.292 | 0.145 |
| Black | 114,925 | 0.622 | 0.841 | -0.118 | | 0.291 | 0.181 |
| NHOPI | 3,006 | 0.607 | 0.821 | -0.132 | | 0.294 | 0.170 |

No group falls below the EEOC 80% threshold. However Black (AIR=0.841) and
NHOPI (AIR=0.821) are close to the threshold and both are statistically and
practically significant per solas-ai.

The FNR disparity is the most legally consequential finding. Black applicants
have FNR=0.181 vs White FNR=0.127 — a 5.4 percentage point gap meaning Black
creditworthy applicants are wrongly denied at a 43% higher rate than White
creditworthy applicants. This is the ECOA-relevant error.

FPR is lower for Black applicants (0.291) than White (0.335) — consistent with
the Impossibility Theorem. When FNR is higher for one group, FPR tends to be
lower. These two parity criteria cannot be simultaneously satisfied when base
rates differ across groups.

Asian applicants have AIR=1.089 — approved at a higher rate than White
applicants. Not a fair lending concern but noted.

---

### 2. Ethnicity

| Group | n | Selection Rate | AIR | ME | Flag | FPR | FNR |
|---|---|---|---|---|---|---|---|
| Not Hispanic (ref) | 709,988 | 0.731 | 1.000 | 0.000 | | 0.329 | 0.128 |
| Hispanic | 122,348 | 0.702 | 0.960 | -0.029 | | 0.319 | 0.130 |

No AIR violation. Hispanic applicants have AIR=0.960, well above the 0.80
threshold. FPR and FNR are nearly identical between groups. Ethnicity alone
does not trigger a fair lending concern in this audit.

---

### 3. Sex

| Group | n | Selection Rate | AIR | ME | Flag | FPR | FNR |
|---|---|---|---|---|---|---|---|
| Male (ref) | 499,597 | 0.747 | 1.000 | 0.000 | | 0.353 | 0.118 |
| Female | 337,955 | 0.696 | 0.932 | -0.051 | | 0.293 | 0.144 |

No AIR violation. Female applicants have AIR=0.932. FNR is higher for Female
(0.144) than Male (0.118) — female creditworthy applicants are wrongly denied
at a higher rate, but the gap is modest and not practically significant.

---

### 4. Age

| Group | n | AIR | Flag | FPR | FNR |
|---|---|---|---|---|---|
| >74 | 34,337 | 0.789 | *** BELOW 0.80 | 0.272 | 0.237 |
| 65-74 | 78,939 | 0.827 | | 0.264 | 0.206 |
| 55-64 | 141,361 | 0.893 | | 0.307 | 0.165 |
| 45-54 | 180,003 | 0.947 | | 0.341 | 0.135 |
| 35-44 (ref) | 201,086 | 1.000 | | 0.354 | 0.106 |
| <25 | 33,760 | 1.058 | | 0.312 | 0.081 |
| 25-34 | 167,051 | 1.068 | | 0.357 | 0.079 |

The only AIR violation in the single-attribute audit. Applicants over 74 have
AIR=0.789 — below the EEOC 80% threshold, statistically significant (p=0.000),
and practically significant per solas-ai. Age is a protected characteristic
under ECOA.

FNR increases monotonically with age — applicants aged 25-34 have FNR=0.079
while applicants over 74 have FNR=0.237, nearly three times higher. Older
applicants are being wrongly denied at dramatically higher rates than younger
applicants. This is the most severe error-rate disparity in the entire audit.

Younger applicants (under 25 and 25-34) have AIR > 1.0 suggesting the model
systematically favors younger applicants, likely reflecting income trajectories
and longer remaining loan terms.

---

### 5. Intersectional Analysis — Race x Sex

| Group | n | AIR | Flag | FPR | FNR |
|---|---|---|---|---|---|
| NHOPI Female | 1,079 | 0.746 | *** BELOW 0.80 | 0.263 | 0.189 |
| Black Female | 60,853 | 0.794 | *** BELOW 0.80 | 0.265 | 0.194 |
| 2+ minority Female | 1,406 | 0.800 | *** BELOW 0.80 | 0.268 | 0.147 |
| AIAN Female | 3,920 | 0.821 | | 0.263 | 0.163 |
| NHOPI Male | 1,927 | 0.836 | | 0.314 | 0.160 |
| Black Male | 54,072 | 0.855 | | 0.321 | 0.168 |
| White Female | 247,419 | 0.944 | | 0.299 | 0.140 |
| White Male (ref) | 391,340 | 1.000 | | 0.360 | 0.118 |
| Asian Female | 23,278 | 1.046 | | 0.352 | 0.074 |
| Asian Male | 44,536 | 1.075 | | 0.366 | 0.061 |

Three groups fall below the 0.80 threshold when race and sex are tested jointly.
None of these violations were visible in the single-attribute analysis. This is
the Impossibility Theorem and intersectionality finding from Lecture 3 — testing
protected attributes separately misses harms concentrated at intersections.

Black Female applicants (n=60,853 — large enough to be statistically reliable)
have AIR=0.794 and FNR=0.194 — the highest FNR of any large group in the entire
audit, nearly double the White Male FNR of 0.118.

---

### 6. Intersectional Analysis — Ethnicity x Sex

No AIR violations. Hispanic Female has the lowest AIR at 0.898, well above 0.80.
Ethnicity does not compound sex disparity — adding ethnicity to the sex dimension
reveals no new violations. This contrasts with the race x sex finding where three
groups fell below 0.80, confirming that race is the primary dimension of disparity
in this model.

---

### Summary of Violations

| Attribute | Group | AIR | Violation |
|---|---|---|---|
| Age | >74 | 0.789 | *** YES |
| Race x Sex | NHOPI Female | 0.746 | *** YES |
| Race x Sex | Black Female | 0.794 | *** YES |
| Race x Sex | 2+ minority Female | 0.800 | *** YES |
| Race | Black | 0.841 | No — but close |
| Race | NHOPI | 0.821 | No — but close |

### Impossibility Theorem Connection
Per Impossibility Theorem, calibration and FPR/FNR parity cannot be simultaneously satisfied
when base rates differ across groups. This audit confirms the tradeoff — optimizing
log loss (calibration) produces FNR disparities for older and minority female
applicants. The accepted residual risk is documented here and the deployment
recommendation requires human review for flagged groups.